# Stage 11 — Mars inference

_Pipeline stage 11 of 14. This is the self-contained deep dive for the stage: narrative + analysis + interpretation, using the canonical `channel_heads` package and on-disk artifacts. Heavy rebuilds run via the `channel-heads` CLI (commands are given inline)._

## Applying the Earth-trained regime models to Mars

For each regime we recompute Mars embeddings with that regime's CNN, apply its combined
XGBoost, and threshold at the regime's operating point — yielding a touching
probability + decision per martian confluence pair. We report per-regime coupling
rates, the probability distributions, and **maps** of where coupling is predicted.

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import channel_heads as ch
from channel_heads.io.paths import PROJECT_ROOT, RESULTS_DIR, EXAMPLE_DEMS
ROOT     = PROJECT_ROOT
MODELS   = ROOT / 'models'
MARS_OUT = ROOT / 'data/Mars/model_outputs'
MARS_IN  = ROOT / 'data/Mars/model_inputs'
MARS_DIR = ROOT / 'data/Mars'
REGIMES_ = ['regA', 'regB', 'regC']
RC = {'regA': '#e41a1c', 'regB': '#377eb8', 'regC': '#4daf4a'}
MODEL_FEATURES = ['orientation_diff_deg','headhead_dist_norm','apex_angle_deg',
                  'strahler_order_diff','proximity_profile_norm']
def _load_op_thr(r):  # per-regime operating threshold, read from disk (auto-tracks retrains)
    p = MODELS / f'optimal_threshold_geom_plus_cnn_emb_{r}.txt'
    return float(p.read_text().strip()) if p.exists() else float('nan')
OP_THR = {r: _load_op_thr(r) for r in REGIMES_}
def _abs(p):
    p = Path(p); return p if p.is_absolute() else ROOT / p
print('channel_heads', ch.__version__, '| root', ROOT)


### 1 · Coupling rates per regime

In [ ]:
rows = []
for r in REGIMES_:
    p = MARS_OUT / f'mars_combined_{r}_predictions.parquet'
    if not p.exists(): continue
    d = pd.read_parquet(p)
    rows.append({'regime': r, 'pairs': len(d), 'networks': d.network_id.nunique(),
                 'touching_%': round(100*d.pred_touching.mean(), 1),
                 'high_conf>=0.8_%': round(100*(d.prob_touching>=0.8).mean(), 1),
                 'mean_prob': round(d.prob_touching.mean(), 3)})
pd.DataFrame(rows)

### 2 · Probability distributions

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for r in REGIMES_:
    p = MARS_OUT / f'mars_combined_{r}_predictions.parquet'
    if p.exists():
        d = pd.read_parquet(p)
        ax.hist(d.prob_touching, bins=40, histtype='step', lw=2, color=RC[r], label=r)
        ax.axvline(OP_THR[r], color=RC[r], ls='--', alpha=0.5)
ax.set_xlabel('P(touching)'); ax.set_ylabel('pairs')
ax.set_title('Mars coupling probability (dashed = operating threshold)'); ax.legend(); plt.show()

### 3 · Maps — predicted coupling across the valley networks (all three regimes)

In [ ]:
import geopandas as gpd
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, r in zip(axes, REGIMES_):
    g = MARS_OUT / f'mars_combined_{r}_predictions.gpkg'
    if g.exists():
        gdf = gpd.read_file(g, layer='pairs')
        gdf.plot(ax=ax, column='prob_touching', cmap='RdYlBu_r', linewidth=0.7,
                 vmin=0, vmax=1, legend=(r == REGIMES_[-1]),
                 legend_kwds={'label': 'P(touching)', 'shrink': 0.6})
        ax.set_title(f'{r} (touching {100*gdf.prob_touching.ge(OP_THR[r]).mean():.0f}%)')
    ax.set_aspect('equal'); ax.axis('off')
plt.suptitle('Predicted channel-head coupling on Mars, by regime'); plt.show()

In [ ]:
# ── Martian valley networks on an orthographic Mars globe (Cartopy) ──────────
# Poster figure: the full valley-network skeleton + this run's regA predicted-
# coupled pairs, drawn on a true Mars sphere (R = 3,396,190 m, the data datum).
import geopandas as gpd
import cartopy.crs as ccrs

MARS_R = 3396190.0  # Mars 2000 sphere radius (m) — matches the GeoPackage datum
mars_globe = ccrs.Globe(semimajor_axis=MARS_R, semiminor_axis=MARS_R, ellipse=None)
mars_geo = f'+proj=longlat +R={MARS_R} +no_defs'        # lon/lat on the Mars sphere

# Full valley-network skeleton (reproject Equirectangular-metres -> lon/lat)
seg = gpd.read_file(MARS_DIR / 'topology/mars_vn_topology_model_ready.gpkg',
                    layer='mars_segments').to_crs(mars_geo)
minx, miny, maxx, maxy = seg.total_bounds
clon, clat = (minx + maxx) / 2.0, (miny + maxy) / 2.0   # frame the globe on the data

proj = ccrs.Orthographic(central_longitude=clon, central_latitude=clat, globe=mars_globe)
pc = ccrs.PlateCarree(globe=mars_globe)
fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={'projection': proj})
ax.set_global(); ax.gridlines(color='0.4', linewidth=0.4, alpha=0.5)
fig.patch.set_facecolor('#0b0b10'); ax.set_facecolor('#0b0b10')

# valley networks (cyan)
ax.add_geometries(seg.geometry, crs=pc, facecolor='none',
                  edgecolor='#00f3ff', linewidth=0.6, alpha=0.9)

# overlay regA predicted-coupled pairs (P >= tuned threshold) in warm orange
note = ''
gA = MARS_OUT / 'mars_combined_regA_predictions.gpkg'
if gA.exists():
    pairs = gpd.read_file(gA, layer='pairs').to_crs(mars_geo)
    coupled = pairs[pairs.prob_touching >= OP_THR['regA']]
    ax.add_geometries(coupled.geometry, crs=pc, facecolor='none',
                      edgecolor='#ff5d2a', linewidth=1.0, alpha=0.95)
    note = f"  -  {len(coupled)} regA coupled pairs (P>={OP_THR['regA']:.2f})"

ax.set_title(f'Martian valley networks - orthographic globe ({len(seg)} segments){note}',
             color='white', fontsize=12, pad=16)
plt.show()


In [ ]:
# --- Interactive Mars globe (Plotly 3D): drag to rotate, scroll to zoom ---
# True 3D Mars sphere (NO Earth basemap): lon/lat -> XYZ on the sphere, valley
# networks drawn on the surface, regA coupled pairs highlighted. Also writes a
# standalone shareable HTML. (Plotly Scattergeo is Earth-only, so we build the
# globe ourselves with Scatter3d + a parametric sphere.)
import numpy as np
import geopandas as gpd
import plotly.graph_objects as go

mars_geo = '+proj=longlat +R=3396190 +no_defs'

def _ll2xyz(lon, lat, r=1.0):
    lo = np.radians(np.asarray(lon, float)); la = np.radians(np.asarray(lat, float))
    return r*np.cos(la)*np.cos(lo), r*np.cos(la)*np.sin(lo), r*np.sin(la)

def _lines_xyz(geoms, r=1.0):
    X, Y, Z = [], [], []
    for g in geoms:
        if g is None:
            continue
        for part in (g.geoms if g.geom_type.startswith('Multi') else [g]):
            x, y, z = _ll2xyz(*part.xy, r)
            X += list(x) + [None]; Y += list(y) + [None]; Z += list(z) + [None]
    return X, Y, Z

seg = gpd.read_file(MARS_DIR / 'topology/mars_vn_topology_model_ready.gpkg',
                    layer='mars_segments').to_crs(mars_geo)
seg['geometry'] = seg.geometry.simplify(0.01)   # ~0.6 km: trims vertices, invisible at globe scale
pairs = gpd.read_file(MARS_OUT / 'mars_combined_regA_predictions.gpkg', layer='pairs').to_crs(mars_geo)
coupled = pairs[pairs.prob_touching >= OP_THR['regA']]

# Mars sphere body (uniform rust colour, no Earth features)
u = np.linspace(0, 2*np.pi, 90); v = np.linspace(0, np.pi, 45)
sx = np.outer(np.cos(u), np.sin(v)); sy = np.outer(np.sin(u), np.sin(v)); sz = np.outer(np.ones_like(u), np.cos(v))
R0 = 0.99
sphere = go.Surface(x=R0*sx, y=R0*sy, z=R0*sz, surfacecolor=np.zeros_like(sx),
                    colorscale=[[0, '#5a2b1b'], [1, '#824733']], showscale=False,
                    hoverinfo='skip', lighting=dict(ambient=0.75, diffuse=0.45, specular=0.04))

nX, nY, nZ = _lines_xyz(seg.geometry, 1.0)
pX, pY, pZ = _lines_xyz(coupled.geometry, 1.0002)  # same radius as networks (avoid parallax)
cen = coupled.geometry.representative_point()
cX, cY, cZ = _ll2xyz(cen.x.values, cen.y.values, 1.0004)

fig = go.Figure([
    sphere,
    go.Scatter3d(x=nX, y=nY, z=nZ, mode='lines', name='valley networks',
                 line=dict(color='#00f3ff', width=2), hoverinfo='skip'),
    go.Scatter3d(x=pX, y=pY, z=pZ, mode='lines', name=f'regA coupled ({len(coupled)})',
                 line=dict(color='#ff5d2a', width=3), hoverinfo='skip'),
    go.Scatter3d(x=cX, y=cY, z=cZ, mode='markers', name='P(touching)',
                 marker=dict(size=2, color='#ff8a5c'), hoverinfo='text',
                 text=[f'net {n} | P={pr:.2f}' for n, pr in zip(coupled.network_id, coupled.prob_touching)]),
])
_noax = dict(visible=False, showgrid=False, zeroline=False, showbackground=False)
fig.update_layout(paper_bgcolor='#0b0b10', font_color='white', height=760,
                  title='Martian valley networks - 3D Mars globe (drag to rotate, scroll to zoom)',
                  margin=dict(l=0, r=0, t=40, b=0),
                  scene=dict(xaxis=_noax, yaxis=_noax, zaxis=_noax, aspectmode='data', bgcolor='#0b0b10'))

out_html = MARS_OUT / 'mars_globe.html'
fig.write_html(str(out_html), include_plotlyjs='cdn')
print(f'standalone 3D globe -> {out_html}  (open in a browser and drag)')
fig.show()


**Takeaway.** Coupling is widespread across the martian valley networks in every
regime (regB highest, regC lowest — the dense/sparse ordering), with the high-confidence
fraction tracking the headline rate. The maps show coupling is not concentrated in a
few networks but distributed across the dissected highlands — consistent with mobile
divides having been a global feature of the valley-forming epoch.